In [2]:

import os
if os.getcwd().endswith('notebooks'):
    print('here')
    os.chdir(r'..')
    os.chdir(r'..')
    os.chdir(r'chess_engine')
import sys
sys.path.append('../')
from chess_engine.src.model.classes.sqlite.models import GamePositionRollup
import numpy as np
from tqdm import tqdm
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import bitboards_to_array, sample_bitboard_dict, bitboards_to_array
from chess_engine.src.model.classes.autoencoder.feature_extractor import get_metadata_from_gpr, sample_metada

from chess_engine.src.model.classes.sqlite.database import  get_db
from chess_engine.src.model.config.config import data_settings
import torch
from torch.utils.data import Dataset, DataLoader

from chess_engine.src.model.config.config import data_settings
import time
import h5py
from torch.utils.data import Dataset, DataLoader

In [3]:
os.getcwd()

'/mnt/c/Users/11049716/git/chess_engine'

In [4]:
sample=None
with next(get_db()) as session:
            for batch_start in range(0, 1):

                records = (session.query(GamePositionRollup)
                          .offset(0)
                          .limit(1)
                          .all())
    
                if not records:
                    break
                for record in records:
                    sample = record

In [5]:
sample_bitboard_values = [getattr(sample, attr) for attr in sample_bitboard_dict.keys()]

In [26]:
bitboards_to_array(sample_bitboard_values).reshape(1,-1).shape

(1, 832)

In [28]:
get_metadata_from_gpr(sample).reshape(1,-1).shape

(1, 4)

In [30]:
np.concatenate((bitboards_to_array(sample_bitboard_values).reshape(1,-1),get_metadata_from_gpr(sample).reshape(1,-1)),axis=1).shape

(1, 836)

numpy.ndarray

In [32]:
def worker_init_fn(worker_id, h5_path):
    global _worker_h5_handles
    print(f"Initializing worker {worker_id} with HDF5 file path: {h5_path}")
    if h5_path:
        _worker_h5_handles[worker_id] = h5py.File(h5_path, 'r', libver='latest', swmr=True)
        print(f"Worker {worker_id} initialized successfully.")


def get_dataloader(h5_path, batch_size=32, shuffle=True, num_workers=4,transform=None):
    dataset = HDF5SingleFileDataset(h5_path,transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        worker_init_fn=lambda worker_id: worker_init_fn(worker_id, dataset.h5_file_path),
        shuffle=shuffle
    )
    return loader


def get_dataloader_full_retrieval_time():
    num_epochs = 1
    train_loader = get_dataloader(data_settings.TrainingDirectory, batch_size=64, shuffle=True, num_workers=0)
    start = time.time()
    i = 0
    for epoch in range(num_epochs):
        for features, labels in train_loader:
            i = i + features.shape[0]
            # print(f"feature shape: {features.shape}, labels shape: {labels.shape}")
            pass
            # features => shape (64, 12, 8, 8)
            # labels   => shape (64, 3)
            # your training logic here...
    end = time.time()
    elapsed_time = end - start
    print(f"total run time: {elapsed_time}, training examples: {i}")
    return elapsed_time



class FlattenTransform:
    def __call__(self, features):
        """
        Flattens the feature tensor into a 1D vector.
        """
        return torch.from_numpy(features).float().view(-1)  # Shape: [832]



class HDF5SingleFileDataset(Dataset):
    """
    A Dataset that reads from one chunked HDF5 file with datasets:
      - "features" of shape (N, num_bitboards, 8, 8)
      - "labels" of shape (N, 3)
    """
    def __init__(self, h5_path, transform=None):
        """
        Args:
            h5_file_path (str): Path to the .h5 file ('data_all.h5').
            transform (callable, optional): A transform to apply to the features.
        """
        super().__init__()
        self.h5_file_path = f"{h5_path}/data_all.h5"
        assert os.path.exists(self.h5_file_path), f"HDF5 file not found at {self.h5_file_path}"
        self.transform = transform

        with h5py.File(self.h5_file_path, 'r', libver='latest', swmr=True) as h5f:
            self.length = h5f['features'].shape[0]



    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Retrieve the worker ID
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is None:
            with h5py.File(self.h5_file_path, 'r') as hf:
                if self.transform:
                    flattened_features = hf["flattened_features"][idx]
                else:
                    features = hf["features"][idx]
                    metadata = hf["metadata"][idx]

        else:
            worker_id = worker_info.id
            if worker_id not in _worker_h5_handles:
                raise KeyError(f"Worker {worker_id} does not have an HDF5 handle. Available: {_worker_h5_handles.keys()}")
            hf = _worker_h5_handles[worker_id]
            if self.transform:
                flattened_features = hf["flattened_features"][idx]
            else:
                features = hf["features"][idx]
                metadata = hf["metadata"][idx]


        # Apply any transform you want to the features
        if self.transform:
            flattened_features = torch.from_numpy(flattened_features).float()
            # metadata = self.transform(metadata)
            # concatenated_output = torch.cat([flattened_features, metadata], dim=0)
            return flattened_features


        metadata_tensor = torch.from_numpy(metadata).float()
        features_tensor = torch.from_numpy(features).float()
        
        return features_tensor, metadata_tensor
            


In [33]:
transform = FlattenTransform()
train_loader = get_dataloader(data_settings.TrainingDirectory,
                              batch_size=64,
                              shuffle=True,
                              num_workers=0,
                              transform=transform)
for outputs  in train_loader:
    print(f"outputs shape:  {outputs.shape}")
    break

outputs shape:  torch.Size([64, 836])


In [148]:
feature_squeezed = feature.squeeze(1)

In [134]:
feature_squeezed.shape

torch.Size([2, 832])